# MTHFR Molecular Dynamics Simulation
## WT Dimer vs Compound Heterozygous Dimer

**Author:** Igor Mihaljko | **ORCID:** 0009-0000-1408-1065

This notebook runs OpenMM molecular dynamics simulations on Google Colab GPU.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (or A100 if available)
2. Upload your CIF files when prompted
3. Results save to Google Drive automatically

**Estimated time:** ~2-4 hours for 100ns on T4 GPU

In [ ]:
# Step 1: Install dependencies (~3 minutes)
!pip install -q openmm pdbfixer mdtraj matplotlib numpy

# Verify GPU
import openmm
print(f'OpenMM version: {openmm.__version__}')
platforms = [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
print(f'Available platforms: {platforms}')
for p in platforms:
    plat = openmm.Platform.getPlatformByName(p)
    print(f'  {p}: speed={plat.getSpeed():.0f}')

# Pick best platform
if 'CUDA' in platforms:
    PLATFORM = 'CUDA'
    print('\n>>> Using CUDA GPU - fastest!')
elif 'OpenCL' in platforms:
    PLATFORM = 'OpenCL'
    print('\n>>> Using OpenCL GPU')
else:
    PLATFORM = 'CPU'
    print('\n>>> WARNING: No GPU found, using CPU (will be slow)')

In [ ]:
# Step 2: Mount Google Drive (saves results permanently)
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/MTHFR_MD_Results'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Results will be saved to: {SAVE_DIR}')

In [ ]:
# Step 3: Download structures from your GitHub repo
!git clone https://github.com/DSMPromo/mthfr-target-validation.git repo

import glob

# Find best CIF files
wt_cifs = sorted(glob.glob('repo/alphafold/results_all/wt_dimer_run1/*model_0.cif'))
cp_cifs = sorted(glob.glob('repo/alphafold/results_all/compound_dimer_run1/*model_0.cif'))

print(f'WT dimer CIF: {wt_cifs[0] if wt_cifs else "NOT FOUND"}')
print(f'Compound dimer CIF: {cp_cifs[0] if cp_cifs else "NOT FOUND"}')

In [ ]:
# Step 4: Configuration
# Change these settings as needed

SIM_LENGTH_NS = 100       # Simulation length in nanoseconds
TEMPERATURE = 300         # Kelvin
IONIC_STRENGTH = 0.15     # Molar (150mM NaCl)
TIMESTEP_FS = 2           # Femtoseconds
SAVE_INTERVAL_PS = 10     # Save frame every 10 ps
PADDING_NM = 1.0          # Water box padding

print(f'Simulation: {SIM_LENGTH_NS} ns')
print(f'Temperature: {TEMPERATURE} K')
print(f'Timestep: {TIMESTEP_FS} fs')
print(f'Save interval: {SAVE_INTERVAL_PS} ps')
print(f'Total steps: {int(SIM_LENGTH_NS * 1e6 / TIMESTEP_FS):,}')
print(f'Total frames: {int(SIM_LENGTH_NS * 1000 / SAVE_INTERVAL_PS):,}')

In [ ]:
# Step 5: Prepare and run simulations
import time
import openmm
from openmm import app, unit
from pdbfixer import PDBFixer
from openmm.app import Modeller
import numpy as np

def prepare_system(cif_path, name):
    """Prepare protein for MD (remove non-protein, add solvent)."""
    print(f'\n{"="*60}')
    print(f'Preparing: {name}')
    print(f'{"="*60}')
    
    fixer = PDBFixer(filename=cif_path)
    
    # Remove FAD and non-standard residues
    modeller = Modeller(fixer.topology, fixer.positions)
    to_delete = [r for r in modeller.topology.residues() 
                 if r.name in ('FAD', 'HOH', 'WAT', 'UNK', 'UNL')]
    if to_delete:
        print(f'  Removing {len(to_delete)} non-protein residues')
        modeller.delete(to_delete)
    
    # Save temp PDB and reload through PDBFixer
    from io import StringIO
    temp = StringIO()
    app.PDBFile.writeFile(modeller.topology, modeller.positions, temp)
    temp.seek(0)
    fixer = PDBFixer(pdbfile=temp)
    
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.4)
    fixer.addSolvent(padding=PADDING_NM * unit.nanometers, 
                     ionicStrength=IONIC_STRENGTH * unit.molar)
    
    # Save prepared structure
    prep_path = f'/content/{name}_prepared.pdb'
    with open(prep_path, 'w') as f:
        app.PDBFile.writeFile(fixer.topology, fixer.positions, f)
    
    print(f'  Atoms: {fixer.topology.getNumAtoms():,}')
    print(f'  Saved: {prep_path}')
    
    return fixer.topology, fixer.positions, prep_path


def run_md(topology, positions, name, length_ns):
    """Run MD simulation."""
    print(f'\n  Setting up {length_ns}ns MD on {PLATFORM}...')
    
    forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    system = forcefield.createSystem(
        topology,
        nonbondedMethod=app.PME,
        nonbondedCutoff=1.0 * unit.nanometers,
        constraints=app.HBonds,
    )
    
    integrator = openmm.LangevinMiddleIntegrator(
        TEMPERATURE * unit.kelvin,
        1.0 / unit.picoseconds,
        TIMESTEP_FS * 0.001 * unit.picoseconds
    )
    
    platform = openmm.Platform.getPlatformByName(PLATFORM)
    simulation = app.Simulation(topology, system, integrator, platform)
    simulation.context.setPositions(positions)
    
    # Energy minimization
    print('  Energy minimization...')
    simulation.minimizeEnergy(maxIterations=1000)
    
    # Equilibration (100ps)
    print('  Equilibrating (100ps)...')
    simulation.context.setVelocitiesToTemperature(TEMPERATURE * unit.kelvin)
    simulation.step(int(100 / (TIMESTEP_FS * 0.001)))  # 100ps
    
    # Production
    traj_path = f'/content/{name}_trajectory.dcd'
    log_path = f'/content/{name}_energy.csv'
    
    save_steps = int(SAVE_INTERVAL_PS / (TIMESTEP_FS * 0.001))
    total_steps = int(length_ns * 1e6 / TIMESTEP_FS)
    
    simulation.reporters.append(app.DCDReporter(traj_path, save_steps))
    simulation.reporters.append(app.StateDataReporter(
        log_path, save_steps,
        step=True, time=True, potentialEnergy=True,
        temperature=True, speed=True
    ))
    
    print(f'  Production: {length_ns}ns ({total_steps:,} steps)...')
    start = time.time()
    
    # Run in 1ns chunks for progress reporting
    chunk_steps = int(1e6 / TIMESTEP_FS)  # 1ns worth of steps
    n_chunks = int(length_ns)
    
    for i in range(n_chunks):
        simulation.step(chunk_steps)
        elapsed = time.time() - start
        speed = (i + 1) / (elapsed / 86400)  # ns/day
        eta = (n_chunks - i - 1) * elapsed / (i + 1)
        print(f'    {i+1}/{n_chunks} ns ({(i+1)/n_chunks*100:.0f}%) | '
              f'Speed: {speed:.1f} ns/day | ETA: {eta/60:.0f} min')
    
    total_time = time.time() - start
    print(f'\n  DONE in {total_time/3600:.1f} hours')
    
    # Save to Google Drive
    import shutil
    for f in [traj_path, log_path, f'/content/{name}_prepared.pdb']:
        shutil.copy(f, SAVE_DIR)
        print(f'  Saved to Drive: {os.path.basename(f)}')
    
    # Save final state
    state = simulation.context.getState(getPositions=True)
    final_path = f'{SAVE_DIR}/{name}_final.pdb'
    with open(final_path, 'w') as f:
        app.PDBFile.writeFile(topology, state.getPositions(), f)
    
    return traj_path


# Run WT dimer
if wt_cifs:
    wt_top, wt_pos, wt_prep = prepare_system(wt_cifs[0], 'wt_dimer')
    run_md(wt_top, wt_pos, 'wt_dimer', SIM_LENGTH_NS)

# Run compound dimer
if cp_cifs:
    cp_top, cp_pos, cp_prep = prepare_system(cp_cifs[0], 'compound_dimer')
    run_md(cp_top, cp_pos, 'compound_dimer', SIM_LENGTH_NS)

In [ ]:
# Step 6: Analysis
import mdtraj as md
import matplotlib.pyplot as plt
import numpy as np

results = {}

for name, label, color in [
    ('wt_dimer', 'Wild-Type Dimer', '#2196F3'),
    ('compound_dimer', 'Compound Het Dimer', '#F44336')
]:
    traj_file = f'/content/{name}_trajectory.dcd'
    top_file = f'/content/{name}_prepared.pdb'
    
    if not os.path.exists(traj_file):
        # Try Google Drive
        traj_file = f'{SAVE_DIR}/{name}_trajectory.dcd'
        top_file = f'{SAVE_DIR}/{name}_prepared.pdb'
    
    if not os.path.exists(traj_file):
        print(f'No trajectory for {name}')
        continue
    
    print(f'\nAnalyzing {name}...')
    traj = md.load(traj_file, top=top_file)
    protein = traj.atom_slice(traj.topology.select('protein'))
    
    rmsd = md.rmsd(protein, protein, 0) * 10  # Angstroms
    rmsf = md.rmsf(protein, protein, 0) * 10
    ca = protein.topology.select('name CA')
    ca_rmsf = rmsf[ca]
    
    results[name] = {
        'label': label, 'color': color,
        'rmsd': rmsd, 'rmsf': ca_rmsf,
        'n_frames': traj.n_frames,
        'time_ns': traj.time[-1] / 1000
    }
    
    print(f'  Frames: {traj.n_frames}')
    print(f'  Time: {traj.time[-1]/1000:.1f} ns')
    print(f'  Mean RMSD: {np.mean(rmsd):.2f} +/- {np.std(rmsd):.2f} A')
    if len(ca_rmsf) > 429:
        print(f'  RMSF @ pos 222: {ca_rmsf[221]:.2f} A')
        print(f'  RMSF @ pos 429: {ca_rmsf[428]:.2f} A')

print('\nAnalysis complete!')

In [ ]:
# Step 7: Generate comparison plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# RMSD over time
for name, r in results.items():
    time_ns = np.arange(len(r['rmsd'])) * SAVE_INTERVAL_PS / 1000
    axes[0,0].plot(time_ns, r['rmsd'], label=r['label'], 
                   color=r['color'], alpha=0.8, linewidth=0.5)
axes[0,0].set_xlabel('Time (ns)', fontsize=12)
axes[0,0].set_ylabel('RMSD (A)', fontsize=12)
axes[0,0].set_title(f'Backbone RMSD Over {SIM_LENGTH_NS}ns', fontsize=14)
axes[0,0].legend(fontsize=11)

# RMSD distribution
for name, r in results.items():
    axes[0,1].hist(r['rmsd'], bins=50, alpha=0.5, color=r['color'],
                   label=f"{r['label']} (mean={np.mean(r['rmsd']):.2f}A)")
axes[0,1].set_xlabel('RMSD (A)', fontsize=12)
axes[0,1].set_ylabel('Count', fontsize=12)
axes[0,1].set_title('RMSD Distribution', fontsize=14)
axes[0,1].legend(fontsize=10)

# Per-residue RMSF
for name, r in results.items():
    residues = np.arange(1, len(r['rmsf']) + 1)
    axes[1,0].plot(residues, r['rmsf'], label=r['label'],
                   color=r['color'], alpha=0.8, linewidth=0.8)
axes[1,0].axvline(x=222, color='red', linestyle='--', alpha=0.5, label='pos 222')
axes[1,0].axvline(x=429, color='purple', linestyle='--', alpha=0.5, label='pos 429')
axes[1,0].set_xlabel('Residue Number', fontsize=12)
axes[1,0].set_ylabel('RMSF (A)', fontsize=12)
axes[1,0].set_title('Per-Residue Flexibility', fontsize=14)
axes[1,0].legend(fontsize=9)

# RMSF difference
if 'wt_dimer' in results and 'compound_dimer' in results:
    wt_rmsf = results['wt_dimer']['rmsf']
    cp_rmsf = results['compound_dimer']['rmsf']
    n = min(len(wt_rmsf), len(cp_rmsf))
    diff = cp_rmsf[:n] - wt_rmsf[:n]
    residues = np.arange(1, n + 1)
    colors = ['#F44336' if d > 0 else '#4CAF50' for d in diff]
    axes[1,1].bar(residues, diff, width=1, color=colors, alpha=0.7)
    axes[1,1].axhline(y=0, color='black', linewidth=0.5)
    axes[1,1].axvline(x=222, color='red', linestyle='--', alpha=0.5)
    axes[1,1].axvline(x=429, color='purple', linestyle='--', alpha=0.5)
    axes[1,1].set_xlabel('Residue', fontsize=12)
    axes[1,1].set_ylabel('RMSF Diff (A)\n(Compound - WT)', fontsize=11)
    axes[1,1].set_title('Flexibility Difference (red = compound more flexible)', fontsize=12)

plt.suptitle(f'MTHFR Molecular Dynamics: WT vs Compound Heterozygous ({SIM_LENGTH_NS}ns)',
             fontsize=16, fontweight='bold')
plt.tight_layout()

# Save
plt.savefig(f'{SAVE_DIR}/md_comparison_{SIM_LENGTH_NS}ns.png', dpi=200, bbox_inches='tight')
plt.savefig('/content/md_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'\nSaved to Google Drive: md_comparison_{SIM_LENGTH_NS}ns.png')

In [ ]:
# Step 8: Summary statistics
print('=' * 60)
print(f'MOLECULAR DYNAMICS SUMMARY ({SIM_LENGTH_NS}ns)')
print('=' * 60)

for name, r in results.items():
    print(f'\n{r["label"]}:')
    print(f'  Frames: {r["n_frames"]}')
    print(f'  Time: {r["time_ns"]:.1f} ns')
    print(f'  Mean RMSD: {np.mean(r["rmsd"]):.2f} +/- {np.std(r["rmsd"]):.2f} A')
    if len(r['rmsf']) > 429:
        print(f'  RMSF @ pos 222: {r["rmsf"][221]:.2f} A')
        print(f'  RMSF @ pos 429: {r["rmsf"][428]:.2f} A')

if 'wt_dimer' in results and 'compound_dimer' in results:
    from scipy import stats
    wt_rmsd = results['wt_dimer']['rmsd']
    cp_rmsd = results['compound_dimer']['rmsd']
    t, p = stats.ttest_ind(wt_rmsd, cp_rmsd)
    print(f'\n--- Statistical Comparison ---')
    print(f'WT mean RMSD: {np.mean(wt_rmsd):.2f} A')
    print(f'Compound mean RMSD: {np.mean(cp_rmsd):.2f} A')
    print(f't-test: t={t:.3f}, p={p:.2e}')
    print(f'Significant: {"YES" if p < 0.05 else "no"}')

print(f'\nAll results saved to Google Drive: {SAVE_DIR}')
print('Download the md_comparison PNG and add to your GitHub repo!')